# Gradients and Initialization

**IMSE 774 · Neural Networks · Week 7 · Sessions 13–14 · October 6 & 8, 2026**

Every code cell from the [lecture note](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/07-gradients-initialization.html), in the order it appears
there. The prose, figures, and exercises live on that page; this notebook is
for running and changing the code.

Run a cell with **Shift+Enter**. Run the setup cell first — the later cells
depend on the names it defines.

Source: [https://github.com/harunpirim/IMSE774-NeuralNetworks](https://github.com/harunpirim/IMSE774-NeuralNetworks)


## Setup


`setup`


In [ ]:
import timeit
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(774)
np.random.seed(774)

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

NDSU_GREEN = "#005643"
NDSU_GOLD  = "#FFC72C"
ACCENT     = "#C1121F"
STEEL      = "#1864AB"

## A Toy Example


`toy-backprop`


In [ ]:
beta  = np.array([0.5, -0.3, 0.2, 1.0])                 # beta_0 ... beta_3
omega = np.array([1.1,  0.7, -0.6, 0.9])                # omega_0 ... omega_3
x_i, y_i = 2.3, 1.4

def forward(beta, omega, x, y):                         # equation (7.8): compute and store
    f0 = beta[0] + omega[0] * x;  h1 = np.sin(f0)
    f1 = beta[1] + omega[1] * h1; h2 = np.exp(f1)
    f2 = beta[2] + omega[2] * h2; h3 = np.cos(f2)
    f3 = beta[3] + omega[3] * h3
    return dict(f0=f0, h1=h1, f1=f1, h2=h2, f2=f2, h3=h3, f3=f3, loss=(f3 - y) ** 2)

s = forward(beta, omega, x_i, y_i)

# Backward pass #1, equations (7.10)-(7.12): each line reuses the one before.
dl_df3 = 2 * (s["f3"] - y_i)
dl_dh3 = omega[3] * dl_df3                              # df3/dh3 = omega_3
dl_df2 = -np.sin(s["f2"]) * dl_dh3                      # dh3/df2 = -sin[f2]
dl_dh2 = omega[2] * dl_df2
dl_df1 = np.exp(s["f1"]) * dl_dh2                       # dh2/df1 = exp[f1]  (already stored as h2)
dl_dh1 = omega[1] * dl_df1
dl_df0 = np.cos(s["f0"]) * dl_dh1                       # dh1/df0 = cos[f0]

# Backward pass #2, equations (7.13)-(7.15).
dl_dbeta  = np.array([dl_df0, dl_df1, dl_df2, dl_df3])
dl_domega = np.array([dl_df0 * x_i, dl_df1 * s["h1"], dl_df2 * s["h2"], dl_df3 * s["h3"]])

# Check 1: the brute-force expression for d l / d omega_0, equation (7.7).
inner = beta[1] + omega[1] * np.sin(beta[0] + omega[0] * x_i)
mid = beta[2] + omega[2] * np.exp(inner)
direct = (-2 * (beta[3] + omega[3] * np.cos(mid) - y_i) * omega[1] * omega[2] * omega[3]
          * x_i * np.cos(beta[0] + omega[0] * x_i) * np.exp(inner) * np.sin(mid))

# Check 2: central finite differences, two extra forward passes per parameter.
eps = 1e-6
def fd(which, k):
    e = np.zeros(4); e[k] = eps
    if which == "beta":
        return (forward(beta + e, omega, x_i, y_i)["loss"] - forward(beta - e, omega, x_i, y_i)["loss"]) / (2 * eps)
    return (forward(beta, omega + e, x_i, y_i)["loss"] - forward(beta, omega - e, x_i, y_i)["loss"]) / (2 * eps)

# Check 3: PyTorch autograd on the same composition.
tb = torch.tensor(beta, requires_grad=True); tw = torch.tensor(omega, requires_grad=True)
t_f = tb[3] + tw[3] * torch.cos(tb[2] + tw[2] * torch.exp(tb[1] + tw[1] * torch.sin(tb[0] + tw[0] * x_i)))
((t_f - y_i) ** 2).backward()

print(f"{'parameter':<10}{'backprop':>13}{'finite diff.':>14}{'autograd':>13}")
for k in range(4):
    print(f"{'beta_' + str(k):<10}{dl_dbeta[k]:>13.8f}{fd('beta', k):>14.8f}{tb.grad[k].item():>13.8f}")
    print(f"{'omega_' + str(k):<10}{dl_domega[k]:>13.8f}{fd('omega', k):>14.8f}{tw.grad[k].item():>13.8f}")
print(f"\nd l / d omega_0 from the brute-force equation (7.7): {direct:.8f}")

## Backpropagation algorithm summary


`deep-backprop`


In [ ]:
def init_network(D, seed=774):
    """Weights and biases for layer widths D = [D_i, D_1, ..., D_K, D_o]."""
    g = np.random.default_rng(seed)
    Omegas = [g.normal(0, np.sqrt(2 / D[k]), (D[k + 1], D[k])) for k in range(len(D) - 1)]
    betas  = [g.normal(0, 0.1, D[k + 1]) for k in range(len(D) - 1)]
    return betas, Omegas

def backprop(betas, Omegas, X, Y):
    """Equations (7.23)-(7.25) for a batch: rows of X are inputs x_i, rows of Y targets.
    Loss is sum_i ||f_K - y_i||^2. Returns gradients summed over the batch."""
    K = len(Omegas) - 1
    f, h = [X @ Omegas[0].T + betas[0]], [X]               # forward pass: store every f_k, h_k
    for k in range(1, K + 1):
        h.append(np.maximum(f[k - 1], 0))
        f.append(h[k] @ Omegas[k].T + betas[k])
    dl_df = 2 * (f[K] - Y)                                  # dl/df_K for least squares, one row per example
    g_beta, g_Omega = [None] * (K + 1), [None] * (K + 1)
    for k in range(K, 0, -1):                               # backward pass
        g_beta[k]  = dl_df.sum(axis=0)                      # sum over the batch
        g_Omega[k] = dl_df.T @ h[k]                         # = sum_i (dl/df_k) h_k^T, an outer product per example
        dl_df = (f[k - 1] > 0) * (dl_df @ Omegas[k])        # I[f_{k-1} > 0] (.) Omega_k^T dl/df_k
    g_beta[0], g_Omega[0] = dl_df.sum(axis=0), dl_df.T @ X
    return g_beta, g_Omega, np.sum((f[K] - Y) ** 2)

D = [5, 40, 40, 40, 3]                                      # three hidden layers, as in figure 7.1
betas, Omegas = init_network(D)
X_b, Y_b = np.random.randn(16, 5), np.random.randn(16, 3)
g_beta, g_Omega, L_ours = backprop(betas, Omegas, X_b, Y_b)

t_betas  = [torch.tensor(b, requires_grad=True) for b in betas]
t_Omegas = [torch.tensor(W, requires_grad=True) for W in Omegas]
h_t = torch.tensor(X_b)
for k, (b, W) in enumerate(zip(t_betas, t_Omegas)):
    f_t = h_t @ W.T + b
    h_t = torch.relu(f_t) if k < len(Omegas) - 1 else f_t
L_torch = ((h_t - torch.tensor(Y_b)) ** 2).sum()
L_torch.backward()

print(f"loss: ours {L_ours:.10f}   PyTorch {L_torch.item():.10f}\n")
print(f"{'layer k':>8}{'shape of dl/dOmega_k':>22}{'max rel. error, Omega':>24}{'max rel. error, beta':>23}")
for k in range(len(Omegas)):
    eW = np.abs(g_Omega[k] - t_Omegas[k].grad.numpy()).max() / np.abs(t_Omegas[k].grad.numpy()).max()
    eb = np.abs(g_beta[k] - t_betas[k].grad.numpy()).max() / np.abs(t_betas[k].grad.numpy()).max()
    print(f"{k:>8}{str(g_Omega[k].shape):>22}{eW:>24.1e}{eb:>23.1e}")

## Cheap in arithmetic, expensive in memory


`fig-cost`


In [ ]:
widths = [16, 32, 64, 128, 256, 512, 1024]
rows = []
for W in widths:
    torch.manual_seed(774)
    net = nn.Sequential(nn.Linear(64, W), nn.ReLU(), nn.Linear(W, W), nn.ReLU(), nn.Linear(W, 10))
    P = sum(p.numel() for p in net.parameters())
    xb, yb = torch.randn(128, 64), torch.randn(128, 10)

    def forward_only():
        with torch.no_grad():
            return ((net(xb) - yb) ** 2).sum()

    def forward_backward():
        net.zero_grad()
        ((net(xb) - yb) ** 2).sum().backward()

    t_f = min(timeit.repeat(forward_only, number=20, repeat=5)) / 20
    t_b = min(timeit.repeat(forward_backward, number=20, repeat=5)) / 20
    rows.append((W, P, t_f, t_b, 2 * P * t_f))
rows = np.array(rows)

plt.figure(figsize=(6.4, 3.7))
plt.loglog(rows[:, 1], rows[:, 2], "o-", color=NDSU_GREEN, lw=2, label="one forward pass")
plt.loglog(rows[:, 1], rows[:, 3], "s-", color=ACCENT, lw=2, label="forward + backpropagation")
plt.loglog(rows[:, 1], rows[:, 4], "^-", color=STEEL, lw=2, label="finite differences ($2P$ forward passes)")
plt.xlabel("number of parameters $P$"); plt.ylabel("seconds per gradient")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

print(f"{'width':>6}{'parameters':>12}{'forward (ms)':>14}{'fwd+bwd (ms)':>14}{'ratio':>7}{'finite diff. (s)':>18}")
for W, P, t_f, t_b, t_fd in rows:
    print(f"{int(W):>6}{int(P):>12,}{t_f * 1e3:>14.3f}{t_b * 1e3:>14.3f}{t_b / t_f:>7.1f}{t_fd:>18.2f}")

`saved-activations`


In [ ]:
from torch.autograd.graph import saved_tensors_hooks

torch.manual_seed(774)
deep = nn.Sequential(*[m for _ in range(10) for m in (nn.Linear(256, 256), nn.ReLU())])
param_ptrs = {p.data_ptr() for p in deep.parameters()}
param_bytes = sum(p.numel() * p.element_size() for p in deep.parameters())

def activation_bytes_saved(net, xb):
    """Bytes autograd stores for the backward pass, excluding the weights themselves."""
    total = [0]
    def pack(t):
        if t.data_ptr() not in param_ptrs:
            total[0] += t.numel() * t.element_size()
        return t
    with saved_tensors_hooks(pack, lambda t: t):
        net(xb).pow(2).sum()
    return total[0]

print("ten layers of Linear(256, 256) + ReLU, float32\n")
print(f"{'batch size':>11}{'stored for backward (MB)':>26}{'parameters (MB)':>18}{'ratio':>8}")
for B in (1, 32, 256, 1024, 4096):
    a = activation_bytes_saved(deep, torch.randn(B, 256))
    print(f"{B:>11,}{a / 1e6:>26.2f}{param_bytes / 1e6:>18.2f}{a / param_bytes:>8.1f}")

## Algorithmic differentiation


`autograd`


In [ ]:
class MyReLU(torch.autograd.Function):
    """A ReLU that 'knows its own derivative', exactly as section 7.4.2 describes."""
    @staticmethod
    def forward(ctx, z_in):
        ctx.save_for_backward(z_in)                     # store what the backward pass will need
        return z_in.clamp(min=0)
    @staticmethod
    def backward(ctx, grad_out):                        # receives dl/dz_out, returns dl/dz_in
        (z_in,) = ctx.saved_tensors
        return grad_out * (z_in > 0)                    # the indicator mask of equation (7.24)

z = torch.randn(6, 4, dtype=torch.float64, requires_grad=True)
print("gradcheck of MyReLU against finite differences:", torch.autograd.gradcheck(MyReLU.apply, (z,)))

lin1, lin2 = nn.Linear(3, 4), nn.Linear(4, 1)
out = lin2(MyReLU.apply(lin1(torch.randn(2, 3)))).pow(2).sum()

print("\nthe graph recorded during the forward pass, walked back from the loss:")
node, depth = out.grad_fn, 0
while node is not None:
    print("   " * depth + type(node).__name__)
    parents = [n for n, _ in node.next_functions if n is not None and type(n).__name__ != "AccumulateGrad"]
    node, depth = (parents[0] if parents else None), depth + 1

## Extension to arbitrary computational graphs


`branching-graph`


In [ ]:
torch.manual_seed(774)
h = torch.randn(4, dtype=torch.float64, requires_grad=True)
A = torch.randn(4, 4, dtype=torch.float64)
B = torch.randn(4, 4, dtype=torch.float64)

out = (torch.relu(A @ h) + torch.tanh(B @ h)).sum()     # h feeds two branches that recombine
out.backward()

branch_1 = A.T @ (A @ h > 0).double()                   # derivative arriving along the ReLU branch
branch_2 = B.T @ (1 - torch.tanh(B @ h) ** 2)           # derivative arriving along the tanh branch
print("autograd dl/dh          :", h.grad.numpy().round(6))
print("sum of the two branches :", (branch_1 + branch_2).detach().numpy().round(6))
print("either branch alone     :", branch_1.detach().numpy().round(6), "(ReLU only)")

## Initialization for the forward pass


`relu-moment`


In [ ]:
rng = np.random.default_rng(774)
print("second moment of ReLU[a] for a ~ Norm[0, sigma^2]  (problem 7.14 predicts sigma^2 / 2)")
for sigma2 in (0.5, 1.0, 4.0):
    a = rng.normal(0, np.sqrt(sigma2), 1_000_000)
    print(f"   sigma^2 = {sigma2:>4}:  E[ReLU[a]^2] = {np.mean(np.maximum(a, 0) ** 2):.4f}   sigma^2 / 2 = {sigma2 / 2:.4f}")

print("\none layer, D_h = 100: the variance multiplier sigma^2_f' / sigma^2_f  (equation 7.30 predicts D_h sigma^2_Omega / 2)")
D_h = 100
f = rng.normal(0, 1, (20_000, D_h))
for sigma2_W in (0.001, 0.01, 0.02, 0.1):
    W = rng.normal(0, np.sqrt(sigma2_W), (D_h, D_h))
    f_next = np.maximum(f, 0) @ W.T
    print(f"   sigma^2_Omega = {sigma2_W:<6}  measured {f_next.var():8.4f}   predicted {D_h * sigma2_W / 2:8.4f}")

## Initialization for both forward and backward pass


`fig-init-variance`


In [ ]:
K, D_h, n = 50, 100, 1000
SIGMA2S = [0.001, 0.01, 0.02, 0.1, 1.0]
COLORS = ["#E07B54", NDSU_GREEN, "#888888", "#7A3B1E", "#7FCFCB"]

def variance_profile(sigma2, seed):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n, D_h, generator=g, dtype=torch.float64)
    Ws = [torch.randn(D_h, D_h, generator=g, dtype=torch.float64) * np.sqrt(sigma2) for _ in range(K)]
    fs, h = [], x
    for W in Ws:                                        # forward pass, equation (7.23) with beta = 0
        fs.append(h @ W.T)
        h = torch.relu(fs[-1])
    fwd = [f.var().item() for f in fs]
    dl_df = 2 * fs[-1]                                  # dl/df_K for loss = sum f_K^2
    bwd = [dl_df.var().item()]
    for k in range(K - 1, 0, -1):                       # backward pass, equation (7.24)
        dl_df = (fs[k - 1] > 0) * (dl_df @ Ws[k])
        bwd.append(dl_df.var().item())
    return np.array(fwd), np.array(bwd[::-1])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
layers = np.arange(1, K + 1)
summary = []
for sigma2, c in zip(SIGMA2S, COLORS):
    runs = [variance_profile(sigma2, seed) for seed in range(10)]
    F = np.array([r[0] for r in runs]); Bk = np.array([r[1] for r in runs])
    for ax, M in ((axes[0], F), (axes[1], Bk)):
        ax.semilogy(layers, np.median(M, 0), color=c, lw=2, label=rf"$\sigma^2_\Omega = {sigma2}$")
        ax.fill_between(layers, M.min(0), M.max(0), color=c, alpha=0.15)
    axes[0].semilogy(layers, F[:, 0].mean() * (D_h * sigma2 / 2) ** (layers - 1), "--", color=c, lw=0.9)
    summary.append((sigma2, np.median(F[:, -1]), np.median(Bk[:, 0]), D_h * sigma2 / 2))
axes[0].set_title("forward pass: variance of $\\mathbf{f}_k$"); axes[0].set_ylabel("variance  (log scale)")
axes[1].set_title(r"backward pass: variance of $\partial \ell / \partial \mathbf{f}_k$")
for ax in axes:
    ax.set_xlabel("layer $k$"); ax.set_ylim(1e-140, 1e185)
axes[0].legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

print(f"{'sigma^2_Omega':>14}{'factor per layer':>18}{'var f_50 (median)':>20}{'var dl/df_1 (median)':>23}")
for sigma2, v_fwd, v_bwd, factor in summary:
    print(f"{sigma2:>14}{factor:>18.2f}{v_fwd:>20.3e}{v_bwd:>23.3e}")

## Initialization in practice


`fig-init-training`


In [ ]:
rng_t = np.random.default_rng(774)
x_t = rng_t.uniform(-1, 1, 512)
y_t = np.sin(3 * x_t) + rng_t.normal(0, 0.1, 512)
X_t = torch.tensor(x_t, dtype=torch.float32)[:, None]
Y_t = torch.tensor(y_t, dtype=torch.float32)[:, None]

def deep_relu_net(init, depth=20, width=64, seed=774):
    torch.manual_seed(seed)
    layers = [nn.Linear(1, width), nn.ReLU()]
    for _ in range(depth - 1):
        layers += [nn.Linear(width, width), nn.ReLU()]
    net = nn.Sequential(*layers, nn.Linear(width, 1))
    if init != "PyTorch default":
        for m in net:
            if isinstance(m, nn.Linear):
                D_in, D_out = m.weight.shape[1], m.weight.shape[0]
                var = {"He": 2 / D_in, "Glorot": 2 / (D_in + D_out), "4 x He": 8 / D_in}[init]
                nn.init.normal_(m.weight, 0, np.sqrt(var))
                nn.init.zeros_(m.bias)
    return net

INITS = {"He": NDSU_GREEN, "Glorot": STEEL, "PyTorch default": ACCENT, "4 x He": NDSU_GOLD}
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.9))
report = {}
for init, c in INITS.items():
    net = deep_relu_net(init)
    ((net(X_t) - Y_t) ** 2).mean().backward()
    gnorms = [m.weight.grad.norm().item() for m in net if isinstance(m, nn.Linear)]
    net.zero_grad()
    opt = torch.optim.SGD(net.parameters(), lr=0.002, momentum=0.9)
    gen = torch.Generator().manual_seed(774)
    history = []
    for epoch in range(150):
        perm = torch.randperm(512, generator=gen)
        for k in range(0, 512, 32):
            b = perm[k:k + 32]
            opt.zero_grad()
            ((net(X_t[b]) - Y_t[b]) ** 2).mean().backward()
            opt.step()
        with torch.no_grad():
            history.append(((net(X_t) - Y_t) ** 2).mean().item())
    history = np.array(history)
    report[init] = (gnorms[0], gnorms[-1], history)
    axes[0].semilogy(np.arange(len(gnorms)), np.nan_to_num(gnorms, posinf=1e30), "o-", color=c, ms=3, lw=1.6, label=init)
    axes[1].semilogy(np.arange(1, 151), np.where(np.isfinite(history), history, np.nan), color=c, lw=2, label=init)
axes[0].set_xlabel("layer (0 = first)"); axes[0].set_ylabel(r"$\|\partial L / \partial \Omega_k\|$ at initialization")
axes[0].set_title("gradient norm by layer, before training"); axes[0].legend(fontsize=8)
axes[1].axhline(0.1 ** 2, color="gray", ls="--", lw=1)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("training MSE"); axes[1].set_title("training"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"variance of the targets (loss of predicting the mean): {Y_t.var().item():.4f}   noise floor: 0.0100\n")
print(f"{'initialization':<17}{'|grad| layer 0':>15}{'|grad| last':>13}{'loss ep. 5':>12}{'ep. 25':>9}{'ep. 150':>9}")
for init, (g0, gL, h) in report.items():
    fmt = lambda v: f"{v:>9.4f}" if np.isfinite(v) else f"{'nan':>9}"
    print(f"{init:<17}{g0:>15.2e}{gL:>13.2e}{fmt(h[4]):>12}{fmt(h[24])}{fmt(h[-1])}")